In [18]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
grewpy.set_config('ud')
# path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/SUD_French-GSD-r2.15"
# grewpy.set_config('sud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

all_matches = corpus.search(grewpy.Request("pattern{X[upos<>PUNCT]}").without("X[InIdiom=Yes]").without("X[Idiom=Yes]").without("X[InTitle=Yes]").without("X[Title=Yes]").without("X[Scrap=Yes]").without("X[Foreign]").without("X[Lang]").without("X-[fixed]->Y").without("Y-[flat:name]->X").without("Y-[goeswith]->X") , clustering_parameter=['X.lemma'])
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value
import json 
# Create a dictionary to map sent_id to sentences for quick lookup
sent_id_to_sentence = {draft[i].meta['sent_id']: draft[i].features for i in range(len(draft))}

match_upos = {}
for key, value in matches.items():
    for m in value:
        match_sent_id = m['sent_id']
        match_node_index = str(m['matching']['nodes']['X'])
        if match_sent_id in sent_id_to_sentence:
            current_sentence_features = sent_id_to_sentence[match_sent_id]
            if match_node_index in current_sentence_features.keys():
                current_token_features = current_sentence_features[match_node_index]
                if 'ExtPos' in current_token_features:
                    match_upos.setdefault((key, current_token_features['ExtPos']), []).append(m)
                else:
                    match_upos.setdefault((key, current_token_features['upos']), []).append(m)
new_match_upos = {}
for key, value in match_upos.items():
    if len(value) > 10:
        new_match_upos[key] = value
match_upos = new_match_upos

for key, value in match_upos.items():
    print(key, len(value))


('€', 'NOUN') 45
('œuvrer', 'VERB') 12
('œuvre', 'NOUN') 115
('œuf', 'NOUN') 19
('œil', 'NOUN') 33
('île', 'NOUN') 124
('être', 'AUX') 9436
('être', 'VERB') 149
('évêque', 'NOUN') 44
('événement', 'NOUN') 50
('évènement', 'NOUN') 20
('évoquer', 'VERB') 29
('évolution', 'NOUN') 39
('évoluer', 'VERB') 65
('éviter', 'VERB') 42
('évidence', 'NOUN') 13
('éventuellement', 'ADV') 12
('éventuel', 'ADJ') 13
('évaluer', 'VERB') 11
('été', 'NOUN') 57
('étudier', 'VERB') 45
('étudiant', 'NOUN') 23
('étude', 'NOUN') 99
('étranger', 'ADJ') 45
('étranger', 'NOUN') 16
('étrange', 'ADJ') 20
('étoile', 'NOUN') 42
('étendre', 'VERB') 36
('état', 'NOUN') 149
('étape', 'NOUN') 28
('étang', 'NOUN') 11
('étage', 'NOUN') 20
('établissement', 'NOUN') 43
('établir', 'VERB') 75
('équiper', 'VERB') 22
('équipement', 'NOUN') 21
('équipe', 'NOUN') 233
('équipage', 'NOUN') 11
('épreuve', 'NOUN') 41
('épouser', 'VERB') 27
('épouse', 'NOUN') 32
('époque', 'NOUN') 100
('épisode', 'NOUN') 49
('énorme', 'ADJ') 11
('énerg

In [19]:
with open("../3. probability_matrix/patterns_all_nodes.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

data = { k : list() for k in match_upos }
for adv, mts in match_upos.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[adv].append(formatted_features)

unique_lemma = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, match_upos in data.items() for m in match_upos for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_lemma) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [20]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(2930, 438)


In [10]:
X

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.00137979, 0.00034495, 0.00034495, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.00103484],
       ...,
       [0.01172818, 0.0051742 , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.00034495, ..., 0.        , 0.        ,
        0.        ],
       [0.00034495, 0.        , 0.        , ..., 0.00103484, 0.        ,
        0.        ]])

In [21]:
X_t = X.transpose()

In [22]:
print(X_t.shape)

(438, 2930)


In [23]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Fit the model for novelty detection
clf = LocalOutlierFactor(n_neighbors=2, contamination=0.1)
clf.fit(X_t)

# Predict outliers 
y_pred = clf.fit_predict(X_t)
n_outliers = np.sum(y_pred == -1)

In [24]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import pandas as pd

# Dimensionality reduction using t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_reduced = tsne.fit_transform(X_t)

# Prepare data for Plotly visualization
data = pd.DataFrame({
    "Component 1": X_reduced[:, 0],
    "Component 2": X_reduced[:, 1],
    "Feature": unique_features,
    "Outlier": ["Outlier" if pred == -1 else "Inlier" for pred in y_pred],
})

# Create a scatter plot with Plotly
fig = go.Figure()

# Add scatter points
fig.add_trace(go.Scatter(
    x=data["Component 1"],
    y=data["Component 2"],
    mode='markers',
    marker=dict(size=10, color=["red" if o == "Outlier" else "blue" for o in data["Outlier"]]),
    text=data["Feature"],  
    customdata=data["Outlier"],  # Inlier/Outlier status for hover
    hovertemplate="Feature: %{text}<br>Status: %{customdata}<extra></extra>"
))

# Update layout
fig.update_layout(
    title="t-SNE Visualization of all features with LOF Outlier Detection",
    xaxis_title="t-SNE Component 1",
    yaxis_title="t-SNE Component 2",
    width=800,
    height=600
)

# Show the plot
fig.show()

print(f"Number of outliers detected: {n_outliers}/{X.shape[1]}")

Number of outliers detected: 44/438


In [25]:
fig.write_html("feature_outliers_UD_Fr_tsne.html")

In [26]:
import pandas as pd
df = pd.DataFrame({
    "Complete_feature": unique_features,
    "Node": [unique_features[i].split(':')[2] for i in range(len(unique_features))],
    "Feature": [unique_features[i].split(':')[3].split('=')[0] for i in range(len(unique_features))],
    "Value": [unique_features[i].split(':')[3].split('=')[1] for i in range(len(unique_features))],
    "Outlier": ["Outlier" if pred == -1 else "Inlier" for pred in y_pred],
    "Score": clf.negative_outlier_factor_
})

In [27]:
print(df.head(10))

            Complete_feature   Node   Feature  Value  Outlier     Score
0  node:X:child:Definite=Def  child  Definite    Def   Inlier -1.191990
1  node:X:child:Definite=Ind  child  Definite    Ind   Inlier -1.548702
2       node:X:child:Emph=No  child      Emph     No   Inlier -1.014987
3      node:X:child:Emph=Yes  child      Emph    Yes   Inlier -1.318021
4    node:X:child:ExtPos=ADJ  child    ExtPos    ADJ  Outlier -5.696938
5    node:X:child:ExtPos=ADP  child    ExtPos    ADP   Inlier -2.881738
6    node:X:child:ExtPos=ADV  child    ExtPos    ADV   Inlier -1.447289
7  node:X:child:ExtPos=CCONJ  child    ExtPos  CCONJ   Inlier -2.970525
8    node:X:child:ExtPos=DET  child    ExtPos    DET  Outlier -7.199861
9   node:X:child:ExtPos=INTJ  child    ExtPos   INTJ   Inlier -1.162883


In [28]:
import plotly.express as px

# Sort the DataFrame by Score
df = df.sort_values(by="Score")

# Add a color column based on the Outlier status
df['Color'] = df['Outlier'].apply(lambda x: 'blue' if x == 'Inlier' else 'red')

# Create the scatter plot
fig = px.scatter(
    df,
    x="Score",
    y="Feature",  # Use "Feature" for the y-axis
    color="Color",  # Use the color column for coloring
    hover_data={"Color": False, "Score": True, "Complete_feature": True},  # Exclude Color from hover text
    color_discrete_map={"blue": "blue", "red": "red"}  # Map colors explicitly
)

# Ensure all features appear on the y-axis
fig.update_layout(
    yaxis=dict(categoryorder="array", categoryarray=df["Feature"].unique()),  # Explicitly set the order of categories
    title="Feature Clustering by Score",
    xaxis_title="Score",
    yaxis_title="Feature",
    height=800,
    showlegend=False  # Hide legend since colors are self-explanatory
)

# Update marker size for better visualization
fig.update_traces(marker=dict(size=10))

# Show the plot
fig.show()

In [29]:
fig.write_html("feature_outliers_UD_Fr_byfeature.html")

In [30]:
import plotly.express as px
# Sort the DataFrame by Score
df = df.sort_values(by="Score")

# Add a color column based on the Outlier status
df['Color'] = df['Outlier'].apply(lambda x: 'blue' if x == 'Inlier' else 'red')

# Create the scatter plot
fig = px.scatter(
    df,
    x="Score",
    y="Node",
    color="Color",  # Use the color column for coloring
    hover_data={"Color": False, "Score": True, "Complete_feature": True},  # Show Complete_feature on hover
    color_discrete_map={"blue": "blue", "red": "red"}  # Map colors explicitly
)

# Update layout for better visualization
fig.update_traces(marker=dict(size=10))  # Adjust marker size
fig.update_layout(
    title="Feature Clustering by Score",
    xaxis_title="Score",
    yaxis_title="Node",
    showlegend=False  # Hide legend since colors are self-explanatory
)

# Show the plot
fig.show()

In [31]:
fig.write_html("feature_outliers_UD_Fr_bynode.html")

In [32]:
fig = go.Figure()
nodes = df['Node'].unique()
for node in nodes:
    inlier_features = df[(df['Node'] == node) & (df['Outlier'] == 'Inlier')]
    outlier_features = df[(df['Node'] == node) & (df['Outlier'] == 'Outlier')]

    total = len(inlier_features) + len(outlier_features)
    inlier_features_percentage = len(inlier_features) / total * 100 if total > 0 else 0
    outlier_features_percentage = len(outlier_features) / total * 100 if total > 0 else 0

    # Group and combine values for inliers
    inlier_hover_text = df[(df['Node'] == node) & (df['Outlier'] == 'Inlier')].groupby('Feature')['Value'].apply(lambda x: " / ".join(x)).reset_index()
    inlier_hover_text = (inlier_hover_text['Feature'] + " = " + inlier_hover_text['Value']).tolist()

    # Group and combine values for outliers
    outlier_hover_text = df[(df['Node'] == node) & (df['Outlier'] == 'Outlier')].groupby('Feature')['Value'].apply(lambda x: " / ".join(x)).reset_index()
    outlier_hover_text = (outlier_hover_text['Feature'] + " = " + outlier_hover_text['Value']).tolist()

    # Combine hover text into a single string for each slice
    hover_text = [
        "<br>".join(inlier_hover_text),  # Inliers hover text
        "<br>".join(outlier_hover_text)  # Outliers hover text
    ]

    fig.add_trace(go.Pie(
        labels = ['Inliers', 'Outliers'],
        values = [len(inlier_features), len(outlier_features)],
        hovertext = [inlier_hover_text, outlier_hover_text],
        hoverinfo = 'text',
        name = node,
        marker = dict(colors = ['blue', 'red']),
    ))

fig.update_layout(
    title="Outlier Detection by Node",
    updatemenus=[
        {
            "buttons": [
                {
                    "label": node,
                    "method": "update",
                    "args": [
                        {"visible": [i == idx for i in range(len(nodes))]},
                        {"title": f"Outlier Detection by Node: {node}"}
                    ]
                }
                for idx, node in enumerate(nodes)
            ],
            "direction": "down",
            "showactive": True,
        }
    ]
)

for i in range(1, len(nodes)):
    fig.data[i].visible = False

fig.show()

In [33]:
fig.write_html("feature_outliers_UD_Fr_bynode_pie.html")

# Feature matrices per POS

What we could do is see how features behave for certain parts of speech - this allows us to reduce dimensionality but also potentially extract more meaningful results

In [34]:
# iterate through the columns of X and keep only the columns where idx2adv[i][1] is a noun
dict_transposed_matrices_pos = {}
pos_tags = list(set([unit[1] for unit in unique_lemma]))
for pos in pos_tags:
    idx = [i for i in range(len(idx2adv)) if idx2adv[i][1] == pos]
    X_pos = X[idx]
    dict_transposed_matrices_pos[pos] = X_pos.transpose()

for pos, matrix in dict_transposed_matrices_pos.items():
    print(f"{pos=}, {matrix.shape=}")

pos='PROPN', matrix.shape=(438, 172)
pos='DET', matrix.shape=(438, 15)
pos='ADJ', matrix.shape=(438, 402)
pos='ADV', matrix.shape=(438, 123)
pos='SCONJ', matrix.shape=(438, 6)
pos='INTJ', matrix.shape=(438, 2)
pos='NUM', matrix.shape=(438, 165)
pos='CCONJ', matrix.shape=(438, 14)
pos='NOUN', matrix.shape=(438, 1396)
pos='ADP', matrix.shape=(438, 35)
pos='AUX', matrix.shape=(438, 3)
pos='PRON', matrix.shape=(438, 40)
pos='X', matrix.shape=(438, 1)
pos='VERB', matrix.shape=(438, 556)


In [35]:
from sklearn.manifold import TSNE
from sklearn.neighbors import LocalOutlierFactor
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Initialize the figure
fig = go.Figure()

# Iterate over each POS and create a t-SNE visualization for its matrix
dropdown_buttons = []
for idx, (pos, matrix) in enumerate(dict_transposed_matrices_pos.items()):
    if pos == 'X':
        continue
    # Fit LOF model
    clf = LocalOutlierFactor(n_neighbors=2, contamination=0.1)
    clf.fit(matrix)
    y_pred = clf.fit_predict(matrix)

    
    # Dimensionality reduction using t-SNE
    if matrix.shape[1] > 2:
        tsne = TSNE(n_components=2, random_state=42)
        X_reduced = tsne.fit_transform(matrix)
    else:
        X_reduced = matrix
    
    # Prepare data for Plotly visualization
    data = pd.DataFrame({
        "Component 1": X_reduced[:, 0],
        "Component 2": X_reduced[:, 1],
        "Feature": unique_features,
        "Outlier": ["Outlier" if pred == -1 else "Inlier" for pred in y_pred],
    })
    
    # Add scatter points for this POS
    fig.add_trace(go.Scatter(
        x=data["Component 1"],
        y=data["Component 2"],
        mode='markers',
        marker=dict(size=10, color=["red" if o == "Outlier" else "blue" for o in data["Outlier"]]),
        name=pos,
        visible=(idx == 0),  # Only the first POS is visible by default
        text=data["Feature"],
        customdata=data["Outlier"],  # Inlier/Outlier status
        hovertemplate="Feature: %{text}<br>Status: %{customdata}<extra></extra>"
    ))
    
    # Add a dropdown button for this POS
    dropdown_buttons.append({
        "label": pos,
        "method": "update",
        "args": [
            {"visible": [i == idx for i in range(len(dict_transposed_matrices_pos))]},
            {"title": f"t-SNE Visualization for POS: {pos}"}
        ]
    })

# Add dropdown menu to the layout
fig.update_layout(
    title="t-SNE Visualization for POS",
    xaxis_title="t-SNE Component 1",
    yaxis_title="t-SNE Component 2",
    width=1000,
    height=600,
    updatemenus=[{
        "buttons": dropdown_buttons,
        "direction": "down",
        "showactive": True,
    }]
)

# Show the plot
fig.show()


In [36]:
fig.write_html("feature_outliers_UD_Fr_bypos_tsne.html")

In [40]:
clf = LocalOutlierFactor(n_neighbors=2, contamination=0.1)
data = []
for pos in pos_tags:
    idx = [i for i in range(len(idx2adv)) if idx2adv[i][1] == pos]
    X_pos = X[idx]
    clf.fit(X_pos.transpose())
    y_pred = clf.fit_predict(X_pos.transpose())
    data.append({
        "POS": pos,
        "Feature": unique_features,
        "Outlier": ["Outlier" if pred == -1 else "Inlier" for pred in y_pred],
        "Score": clf.negative_outlier_factor_
    })
    

# for pos, matrix in dict_transposed_matrices_pos.items():
#     print(f"{pos=}, {matrix.shape=}")

# data = []
# for idx, (pos, matrix) in enumerate(dict_transposed_matrices_pos.items()):
#     clf.fit(matrix)
#     y_pred = clf.fit_predict(matrix)
#     data.append({
#         "POS": pos,
#         "Outliers": np.sum(y_pred == -1),
#         "Inliers": np.sum(y_pred == 1),
#     })

In [42]:
import pandas as pd

df = pd.DataFrame(data)

In [54]:
#Explode the lists into separate rows
df = df.explode(["Feature", "Outlier", "Score"])

# Map "Inlier" and "Outlier" to colors
df["Color"] = df["Outlier"].map({"Inlier": "blue", "Outlier": "red"})
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["Log_Score"] = np.log1p(df["Score"].abs())
# Create the scatter plot
fig = px.scatter(
    df,
    x="Log_Score",
    y="POS",
    color="Color",
    color_discrete_map={"blue": "blue", "red": "red"},
    hover_data=["Feature", "Outlier"]
)

# Update layout for better visualization
fig.update_layout(
    title="Log-Transformed Feature Scores by POS",
    xaxis_title="Log of Absolute Score",
    yaxis_title="POS",
    legend_title="Outlier Status",
    height=600
)

# Show the plot
fig.show()

In [55]:
fig.write_html("feature_outliers_UD_Fr_bypos_axis.html")

In [44]:
df.head(5)

,POS,Feature,Outlier,Score
0,PROPN,"[node:X:child:Definite=Def, node:X:child:Defin...","[Inlier, Outlier, Outlier, Inlier, Inlier, Inl...","[-0.9761335105194815, -4551498.8160825865, -34..."
1,DET,"[node:X:child:Definite=Def, node:X:child:Defin...","[Inlier, Inlier, Inlier, Inlier, Inlier, Inlie...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."
2,ADJ,"[node:X:child:Definite=Def, node:X:child:Defin...","[Inlier, Inlier, Inlier, Inlier, Inlier, Outli...","[-1.6962770332622978, -2.856723367359188, -1.1..."
3,ADV,"[node:X:child:Definite=Def, node:X:child:Defin...","[Outlier, Inlier, Outlier, Inlier, Inlier, Inl...","[-8.772772202930437, -4.123104985411567, -4040..."
4,SCONJ,"[node:X:child:Definite=Def, node:X:child:Defin...","[Inlier, Inlier, Inlier, Inlier, Inlier, Inlie...","[-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1...."


In [4]:
import numpy as np

# Example numpy array x
x = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# Calculate the sum of each column
row_sums = x.sum(axis=1)
print(row_sums)

# # Create array y by dividing each element of x by the sum of its column
# y = x / row_sums[:, np.newaxis]

# print(y)

[ 6 15 24]
